In [160]:
import json
import pandas as pd

with open("../../dataset/train_sentiment.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print(df.head())
print(df["sentiment"].value_counts())

                                                text sentiment
0  มาตามนัด 🚮🗑️ . . #เขตพญาไท #กรุงเทพมหานคร @สาย...  positive
1  &#9888; แยกราชเทวีจะรื้อน้ำพุกี่โมง? &#128591;...  negative
2  &#128308;สด!"ไอซ์ รักชนก" แถลงข่าวการประมูลคลื...  negative
3  &#127808;#เขตหนองจอก >>ติดตามการดำเนินงานโครงก...   neutral
4  ไทวัสดุ ส่งช่างมือ 1 วีฟิกซ์ ร่วมฟื้นฟู ปรับภู...  positive
sentiment
positive    3000
negative    3000
neutral     3000
Name: count, dtype: int64


In [161]:
import sys
import os
sys.path.append(os.path.abspath("../../"))

In [162]:
import re, html

def clean_text(text):
    text = html.unescape(text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'@\S+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    text = text.replace('\n', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text.strip()

# df["text"] = df["text"].apply(clean_text)

text = df["text"]
sentiment = df["sentiment"]

In [163]:
from sklearn.model_selection import train_test_split

text_train, text_val, sentiment_train, sentiment_val = train_test_split(
    text, sentiment, test_size=0.1, random_state=42
)

In [164]:
# from sklearn.feature_extraction.text import TfidfVectorizer
# from pythainlp.tokenize import word_tokenize
# from models.preprocess import tokenizer

# # def tokenize(text):
# #     return word_tokenize(text, engine="newmm")

# vectorizer = TfidfVectorizer(
#     tokenizer=tokenizer,
#     ngram_range=(1,2),
# )

# text_train_tfidf = vectorizer.fit_transform(text_train)
# text_val_tfidf = vectorizer.transform(text_val)

In [165]:
# from sklearn.svm import LinearSVC
# from sklearn.metrics import classification_report, accuracy_score


# model = LinearSVC(C=1.5)
# model.fit(text_train_tfidf, sentiment_train)

# y_pred = model.predict(text_val_tfidf)
# print(classification_report(sentiment_val, y_pred))
# print("Accuracy:", accuracy_score(sentiment_val, y_pred))

In [166]:
# import pickle

# with open("sentiment_model_new.pkl", "wb") as f:
#     pickle.dump((vectorizer, model), f)

In [167]:
from models.preprocess import tokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

word_vectorizer = TfidfVectorizer(
    tokenizer=tokenizer,
    ngram_range=(1, 2),
    sublinear_tf=True,
    max_df=0.9,
    min_df=1
)

char_vectorizer = TfidfVectorizer(
    analyzer='char_wb', 
    ngram_range=(1, 3), 
    sublinear_tf=True,
    max_df=0.9,
    min_df=3
)

combined_features = FeatureUnion([
    ("word_tfidf", word_vectorizer),
    ("char_tfidf", char_vectorizer)
], transformer_weights={
    "word_tfidf": 1.0,
    "char_tfidf": 0.5
})

model = LinearSVC(C=1.5, max_iter=3000, random_state=42)

pipeline = Pipeline([
    ('features', combined_features),
    ('classifier', model)
])


print("Training Tuned LinearSVC...")
pipeline.fit(text_train, sentiment_train)

y_pred = pipeline.predict(text_val)

print("Accuracy:", accuracy_score(sentiment_val, y_pred))
print(classification_report(sentiment_val, y_pred))

Training Tuned LinearSVC...


c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Accuracy: 0.8044444444444444
              precision    recall  f1-score   support

    negative       0.84      0.85      0.84       294
     neutral       0.77      0.73      0.75       300
    positive       0.81      0.83      0.82       306

    accuracy                           0.80       900
   macro avg       0.80      0.80      0.80       900
weighted avg       0.80      0.80      0.80       900



In [168]:
# # 5. เทรนและประเมินผล
# print("Training model (Might take a little longer due to FeatureUnion...)")
# pipeline.fit(text_train, sentiment_train)

# y_pred = pipeline.predict(text_val)

# print("Accuracy:", accuracy_score(sentiment_val, y_pred))
# print(classification_report(sentiment_val, y_pred))